## Hybrid Retrieval vs ContextualCompressor Retriever

In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_chroma import Chroma
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers.document_compressors import (
    LLMChainExtractor,
    EmbeddingsFilter,
    DocumentCompressorPipeline
)

from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

In [3]:
filename= Path("/home/tacktile/SIM_Support-ChatBot/knowledge_base/RAG_KB_bot.md")

with open(filename, "r") as f:
    markdown_text = f.read()

markdown_text

'# Simple Invoice Manager (SIM) — In App Features Documentation\n# Company / Business Setup\n\n## Definition\nA feature that lets you set up and manage your business details either during first-time onboarding or later from the app settings (Primary Settings, Invoice Header & Footer).\n\n## When & Who\n- Filled by the business owner during initial app setup (onboarding) and updated anytime later from settings whenever business information changes.\n\n## Flows / Navigation:\n### A. Flow for Onboarding: \n\n1. Install the App you can see Onboarding Screen.\n2. Upload your **company logo** and **owner signature** (can be added manually).\n3. Fill in the required business details (see Field Reference below).\n4. Set your preferences (country, currency, financial year, date format).\n5. Tap **Save / Update**. Wait a moment for the database to update.\n6. Note:- [If you want access this information inside the app]\n\n    1. Go to Settings >> Primary Setting (Transaction No., Tax Identificati

In [4]:
headers_to_split_on= [
    ("#", "Header_1"),
    ("##", "Header_2"),
    ("###", "Header_3")
]

text_splitters = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, 
                                            strip_headers=False)

chunks = text_splitters.split_text(markdown_text)

In [5]:
len(chunks)

272

### Hybrid Approach

In [6]:
!pip install rank-bm25

In [7]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding = embedding,
    collection_name='hybrid_search'
)

chroma_retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k': 4}
)

bm25_retriever = BM25Retriever.from_documents(
    documents=chunks,
    k=2,
    bm25_variant='plus'
)

In [8]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[chroma_retriever, bm25_retriever],
    weights=[0.8,0.2]
)

In [9]:
query = "How to create Invoice?"

bm25_results = bm25_retriever.invoke(query)
chroma_results = chroma_retriever.invoke(query)
ensemble_results = ensemble_retriever.invoke(query)

print("=== BM25 Only (keyword match) ===")
for i, doc in enumerate(bm25_results, 1):
    print(f"  [{i}] topic={doc.metadata['Header_1']}: {doc.page_content}")
print()

print("=== ChromaDB Only (semantic match) ===")
for i, doc in enumerate(chroma_results, 1):
    print(f"  [{i}] topic={doc.metadata['Header_1']}: {doc.page_content}")
print()

# Ensemble combines both — surfaces keyword matches AND semantic matches
print("=== Ensemble / Hybrid (keyword + semantic) ===")
for i, doc in enumerate(ensemble_results, 1):
    print(f"  [{i}] topic={doc.metadata['Header_1']}: {doc.page_content}")

=== BM25 Only (keyword match) ===
  [1] topic=Delivery Note: ## FAQs  
**Q: What is a Delivery Note in SIM?**
A: A delivery note is a shipping document that accompanies goods during transit. It confirms what was dispatched, in what quantity, and to where. It is not a payment request — it is proof of dispatch and delivery.  
**Q: How to enable the Delivery Note feature in SIM?**
A: Go to Settings → Enable / Disable Features → Delivery Note → Enable.  
**Q: How to create a delivery note from an invoice?**
A: Go to the Invoice List → tap on the invoice → tap Make Delivery Note.  
**Q: How to create a standalone delivery note and convert it to an invoice later?**
A: Go to Side Menu → Delivery Note → Create Delivery Note. Later, go to Delivery Note List → tap on a Pending Delivery Note → tap Make Invoice.  
**Q: What is the difference between a Delivery Note and an E-Way Bill?**
A: A delivery note is an internal business document confirming dispatch. An E-Way Bill is a government-mandated c

## ContextualCompressorRetriever

In [10]:
base_retriever = vectorstore.as_retriever(search_kwargs = {'k':3})

compressor = LLMChainExtractor.from_llm(llm)

compressor_retriever = ContextualCompressionRetriever(
    base_compressor= compressor,
    base_retriever= base_retriever
)

In [11]:
query = "How to create Invoice?"

final_result = compressor_retriever.invoke(query)

print(f"Query: {query}")
for i, doc in enumerate(final_result, 1):
    print(f"[{i}] | Metadata: {doc.metadata} | Content: {doc.page_content}\n")

Query: How to create Invoice?
[1] | Metadata: {'Header_1': 'Invoices', 'Header_2': 'Video Tutorial'} | Content: How to Create an Invoice: https://www.youtube.com/watch?v=SurVRuMIkUg

[2] | Metadata: {'Header_1': 'Invoices', 'Header_2': 'Navigation'} | Content: Dashboard → Side Menu (☰) → Invoice → Add New Invoice



In [12]:
embeddings_filter = EmbeddingsFilter(embeddings= embedding, similarity_threshold= 0.54)

compressor_retriever_embedding = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=base_retriever
)

In [14]:
final_result_emb = compressor_retriever_embedding.invoke(query)
final_result_emb

[_DocumentWithState(metadata={'Header_1': 'Invoices', 'Header_2': 'Video Tutorial'}, page_content='## Video Tutorial  \nHow to Create an Invoice: https://www.youtube.com/watch?v=SurVRuMIkUg', state={'embedded_doc': [-0.03515625, 0.0194244384765625, -0.07794189453125, -0.00487518310546875, 0.005329132080078125, -0.0635986328125, -0.021820068359375, -0.02294921875, 0.0254669189453125, 0.01006317138671875, 0.031494140625, -0.028472900390625, -0.031158447265625, -0.0281982421875, 0.04498291015625, 0.02911376953125, -0.01422882080078125, 0.046875, -0.002262115478515625, 0.0474853515625, 0.0216064453125, 0.051116943359375, 4.5299530029296875e-05, 0.0025157928466796875, 0.01496124267578125, -0.0386962890625, -0.0053558349609375, 0.019287109375, -0.01216888427734375, -0.0302734375, 0.0007724761962890625, -0.02435302734375, 0.00667572021484375, -0.0026035308837890625, -0.0037250518798828125, 0.01116943359375, 0.0472412109375, 0.00896453857421875, -0.0163116455078125, 0.0130767822265625, -0.0236

In [19]:
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[embeddings_filter, compressor]
)
pipeline_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=base_retriever
)

In [20]:
pipeline_result = pipeline_retriever.invoke(query)
print(f"Query: {query}")
for i, doc in enumerate(pipeline_result, 1):
    print(f"[{i}] | Metadata: {doc.metadata} | Content: {doc.page_content}\n")

Query: How to create Invoice?
[1] | Metadata: {'Header_2': 'Video Tutorial', 'Header_1': 'Invoices'} | Content: How to Create an Invoice: https://www.youtube.com/watch?v=SurVRuMIkUg

[2] | Metadata: {'Header_2': 'Navigation', 'Header_1': 'Invoices'} | Content: Dashboard → Side Menu (☰) → Invoice → Add New Invoice

